# 02 — Train XGBoost severity classifier

ASHA-AI Plan 2.0 · Role C

Three-class XGBoost on the 50-symptom multi-hot + demographic + vitals feature vector. Produces:
- `ml/models/xgboost_v1.pkl`
- `ml/models/xgboost_v1_metadata.json`

Member B's backend (`app/ml/classifier.py`) loads these at startup.

**Target metric:** test macro-F1 ≥ 0.70 (sanity check; the load-bearing number is **emergency-bucket recall = 100%** on the eval suite — see notebook 04).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from train_and_eval import train
model, cols, meta = train(seed=42, n_per_class=1500)
print(f"test_accuracy={meta['test_accuracy']:.3f}  test_macro_f1={meta['test_macro_f1']:.3f}")


In [ ]:
import json, pandas as pd
# Inspect feature importance (top 20)
fi = pd.Series(model.feature_importances_, index=cols).sort_values(ascending=False).head(20)
fi.to_frame('importance').round(4)


In [ ]:
# Spot-check the model on a handful of inputs
from train_and_eval import triage_one
for txt, hist, age in [
    ('severe chest pain radiating to my left arm, sweating', ['diabetes'], 60),
    ('runny nose and mild cough for 2 days, no fever', [], 30),
    ('my child has fever 39.5 and is very lethargic', [], 3),
    ('I dont want to live anymore I have been thinking about ending it', [], 19),
]:
    r = triage_one(model, cols, txt, age=age, history=hist)
    print(f'\n{age}y {hist!r}:  {txt!r}\n  -> {r["level"]}  flags={[f[0] for f in r["flags"]]}  probs={ {k: round(v,2) for k,v in r["probs"].items()} }')
